# LED OCR ablation, phase 3: generate and score (Colab T4)
Protocol `led-ocr-ablation-261110-v1`. Same paths and SET as the extract notebook. Summary text stays private; no comparison is computed here.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json
APP = '/content/multimodal-lecture-summarizer'   # app repo (separate repo)
BENCH = '/content/benchmark'                     # benchmark repo (separate repo)
APP_COMMIT = '5e26ea799de9ac7e5a934d684ed4a031ff1202e4'  # frozen in protocol.json
# benchmark repo is private: add a Colab secret named GITHUB_TOKEN (read access to the repo) and enable notebook access.
try:
    from google.colab import userdata
    _tok = userdata.get('GITHUB_TOKEN')
except Exception:
    _tok = ''
def _clone(repo, dest):
    if os.path.isdir(dest): return
    url = f'https://{_tok}@github.com/multimodal-lecture-summarizer/{repo}.git' if _tok else f'https://github.com/multimodal-lecture-summarizer/{repo}.git'
    code = os.system(f'GIT_TERMINAL_PROMPT=0 git clone -q {url} {dest}')
    assert code == 0 and os.path.isdir(dest), f'clone of {repo} failed (private repo? set the GITHUB_TOKEN Colab secret)'
_clone('benchmark', BENCH); _clone('multimodal-lecture-summarizer', APP)
assert os.system(f'git -C {APP} checkout -q {APP_COMMIT}') == 0, 'frozen app commit not found on the remote'  # scripts refuse any other commit
WORK = '/content/drive/MyDrive/led-ocr-ablation-261110'
SET = 'smoke'   # 'smoke' dry run first, then 'cohort'
PROTOCOL = f'{BENCH}/experiments/registry/protocols/led-ocr-ablation-261110-v1'
hashes = json.load(open(f'{PROTOCOL}/hashes.json'))['sha256']
for name, expected in hashes.items():
    assert hashlib.sha256(open(f'{PROTOCOL}/{name}', 'rb').read()).hexdigest() == expected, name

In [ ]:
!pip -q install -r {APP}/ai_workers/requirements.txt
assert _exit_code == 0, f'step failed with exit code {_exit_code}; read the output above'
!pip -q install -r {BENCH}/requirements.txt
assert _exit_code == 0, f'step failed with exit code {_exit_code}; read the output above'
!pip -q install 'minicheck @ git+https://github.com/Liyan06/MiniCheck.git@b58b9fa69acbd1015ec970fa65dd752413a053d2'
!nvidia-smi -L


In [ ]:
# 3a. Generate A then D per video. Resumable; an error cell is final.
!cd {BENCH} && python -m experiments.scripts.led_ocr_ablation_generate --app-root {APP} --out-dir {WORK}/run --set {SET} --protocol-dir {PROTOCOL}
assert _exit_code == 0, f'step failed with exit code {_exit_code}; read the output above'


In [ ]:
# Audit before scoring: both arms present, same input hash, arm A source is the plain transcript.
import sys; sys.path.insert(0, BENCH)
from experiments.scripts.led_ocr_ablation_generate import source_check
from experiments.scripts.led_ocr_ablation_extract import read_rows
from pathlib import Path
cells = read_rows(Path(f'{WORK}/run/cells_receipt_{SET}.jsonl'))
ok = [c for c in cells if c.get('status') == 'ok']
print(len(cells), 'cells,', len(cells) - len(ok), 'error cells')
print(source_check(ok) or 'source audit clean')
print('D cells with no OCR in the input:', [c['id'] for c in ok if c['arm'] == 'D' and c['ocr_words'] == 0])

In [ ]:
# 3b. Score. Prints no comparison. Needs the references (gigant/tib at the pinned revision) or --references <private json>.
!cd {BENCH} && python -m experiments.scripts.led_ocr_ablation_score --out-dir {WORK}/run --set {SET}
assert _exit_code == 0, f'step failed with exit code {_exit_code}; read the output above'
